
# Meta-Prompting Experiment: Direct Prompting vs. Meta-Prompting

## Objective

This notebook implements a small **Meta-Refiner** experiment for Phase 2: *The Engineering of Reasoning*.

We compare two approaches for handling underspecified user queries:

1. **Direct Prompting** — send the user's request directly to the Task LLM.
2. **Meta-Prompting** — first use a Meta LLM to generate/refine a prompt, then use that prompt with the Task LLM.

### Experiment

```text
                    User Task
                       |
             +---------+---------+
             |                   |
             v                   v
       Direct Prompt       Meta-Prompting
             |                   |
             |              Meta LLM
             |                   |
             |            Optimized Prompt
             |                   |
             |                   v
             |               Task LLM
             |                   |
             +---------+---------+
                       |
                       v
                Compare Results
```

### What we want to learn

- Does Meta-Prompting improve the quality of responses?
- Does it identify ambiguity and missing requirements better?
- Does it produce more actionable answers?
- What additional latency and token/cost overhead does the Meta LLM introduce?

> **Experimental principle:** Use the same model and final-task configuration for the Direct and Meta-Prompting approaches so that the comparison focuses on the effect of prompt refinement.



## 1. Setup

Install the OpenAI Python SDK if it is not already installed.

The SDK reads the `OPENAI_API_KEY` environment variable automatically.

For example:

```bash
pip install -U openai
```

Set your API key before running the notebook:

```bash
export OPENAI_API_KEY="your-api-key"
```

On Windows PowerShell:

```powershell
$env:OPENAI_API_KEY="your-api-key"
```


In [4]:

# Uncomment if the OpenAI SDK is not installed:
%pip install -U openai
%pip install -U pandas

import os
import time
from openai import OpenAI

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "OPENAI_API_KEY is not set. "
        "Set it in your environment before running this notebook."
    )

client = OpenAI()

# Change this if you want to experiment with another available model.
MODEL = "gpt-5"

print(f"Using model: {MODEL}")


Using model: gpt-5



## 2. Direct Prompting

In the baseline approach, the user's request is sent directly to the Task LLM.

```text
User Task
   |
   v
Task LLM
   |
   v
Answer
```


In [6]:

def direct_prompt(user_task: str) -> str:
    """Send the user's task directly to the LLM."""

    response = client.responses.create(
        model=MODEL,
        instructions=(
            "You are a helpful expert. "
            "Answer the user's request directly and provide "
            "a practical, actionable response."
        ),
        input=user_task,
        max_output_tokens=1500,
    )

    return response.output_text



## 3. Meta-Prompting

The Meta-Prompting approach uses two LLM calls:

1. **Meta LLM** analyzes the user's request and generates an optimized prompt.
2. **Task LLM** executes the optimized prompt.

```text
User Task
   |
   v
Meta LLM
   |
   v
Optimized Prompt
   |
   v
Task LLM
   |
   v
Answer
```

The Meta LLM is instructed to consider:

- the user's main goal
- ambiguity and missing requirements
- role definition
- task description
- output format
- constraints
- reasonable assumptions


In [7]:

META_INSTRUCTION = """
You are an expert prompt engineer.

Transform the user's task into a high-quality prompt
for another LLM to execute.

The generated prompt should:

1. Identify the user's main goal.
2. Identify important missing or ambiguous requirements.
3. Define an appropriate role for the Task LLM.
4. Provide a clear task description.
5. Define the expected output format.
6. Include relevant constraints.
7. Make reasonable assumptions when clarification is not essential.

Return ONLY the optimized prompt.

User task:
"""


def generate_meta_prompt(user_task: str) -> str:
    """Use an LLM to transform the user's task into an optimized prompt."""

    response = client.responses.create(
        model=MODEL,
        instructions=META_INSTRUCTION,
        input=user_task,
        max_output_tokens=1000,
    )

    return response.output_text


def meta_prompt(user_task: str) -> tuple[str, str]:
    """
    Generate an optimized prompt and then use it
    to solve the original task.
    """

    # Step 1: Meta LLM generates the prompt.
    optimized_prompt = generate_meta_prompt(user_task)

    # Step 2: Task LLM executes the generated prompt.
    response = client.responses.create(
        model=MODEL,
        instructions=optimized_prompt,
        input=user_task,
        max_output_tokens=1500,
    )

    return optimized_prompt, response.output_text



## 4. Run a Single Comparison

Let's start with one deliberately underspecified engineering request.

> "My API is slow. Can you help me fix it?"

The goal is not to assume that one approach will always be better. We want to inspect the outputs and evaluate the differences.


In [4]:

task = "My API is slow. Can you help me fix it?"

start = time.perf_counter()
direct_result = direct_prompt(task)
direct_latency = time.perf_counter() - start

start = time.perf_counter()
optimized_prompt, meta_result = meta_prompt(task)
meta_latency = time.perf_counter() - start

print("=" * 80)
print("USER TASK")
print("=" * 80)
print(task)

print("\n" + "=" * 80)
print("DIRECT PROMPTING RESULT")
print("=" * 80)
print(direct_result)

print("\n" + "=" * 80)
print("GENERATED META-PROMPT")
print("=" * 80)
print(optimized_prompt)

print("\n" + "=" * 80)
print("META-PROMPTING RESULT")
print("=" * 80)
print(meta_result)

print("\n" + "=" * 80)
print("LATENCY")
print("=" * 80)
print(f"Direct: {direct_latency:.2f}s")
print(f"Meta:   {meta_latency:.2f}s")


USER TASK
My API is slow. Can you help me fix it?

DIRECT PROMPTING RESULT


GENERATED META-PROMPT


META-PROMPTING RESULT
Absolutely. Let’s make this systematic so we find the real bottleneck and fix it, not guess.

First, a few questions so I can tailor advice:


LATENCY
Direct: 22.82s
Meta:   47.00s



## 5. Test Multiple Underspecified Queries

A single example is not enough to draw conclusions.

We will test several engineering queries that intentionally leave requirements unspecified.


In [8]:

test_cases = [
    "My API is slow. Can you help me fix it?",
    "Design a scalable notification system.",
    "Make my Lambda function faster.",
    "Design a CRM integration.",
    "Analyze our customer data.",
    "Our database is slow. What should we do?",
]

print(f"Number of test cases: {len(test_cases)}")
for i, task in enumerate(test_cases, start=1):
    print(f"{i}. {task}")


Number of test cases: 6
1. My API is slow. Can you help me fix it?
2. Design a scalable notification system.
3. Make my Lambda function faster.
4. Design a CRM integration.
5. Analyze our customer data.
6. Our database is slow. What should we do?



## 6. Run the Benchmark

For each test case we record:

- Direct Prompting response
- Meta-Prompting response
- Generated optimized prompt
- Direct latency
- Meta-Prompting latency

The results are kept in memory so that we can inspect and evaluate them later.


In [9]:
from dataclasses import dataclass

@dataclass
class ExperimentResult:
    task: str
    direct_result: str
    optimized_prompt: str
    meta_result: str
    direct_latency: float
    meta_latency: float


def run_experiment(test_cases: list[str]) -> list[ExperimentResult]:
    results = []

    for index, task in enumerate(test_cases, start=1):
        print(f"Running {index}/{len(test_cases)}: {task}")

        start = time.perf_counter()
        direct_result = direct_prompt(task)
        direct_latency = time.perf_counter() - start

        start = time.perf_counter()
        optimized_prompt, meta_result = meta_prompt(task)
        meta_latency = time.perf_counter() - start

        results.append(
            ExperimentResult(
                task=task,
                direct_result=direct_result,
                optimized_prompt=optimized_prompt,
                meta_result=meta_result,
                direct_latency=direct_latency,
                meta_latency=meta_latency,
            )
        )

    return results


results = run_experiment(test_cases)

print(f"\nCompleted {len(results)} experiments.")


Running 1/6: My API is slow. Can you help me fix it?
Running 2/6: Design a scalable notification system.
Running 3/6: Make my Lambda function faster.
Running 4/6: Design a CRM integration.
Running 5/6: Analyze our customer data.
Running 6/6: Our database is slow. What should we do?

Completed 6 experiments.



## 7. Compare the Results

The table below gives a quick view of the latency overhead.

For response quality, the experiment should use a human evaluation or a separate evaluator rather than assuming that the longer response is automatically better.


In [12]:
import pandas as pd

summary = pd.DataFrame(
    [
        {
            "Task": r.task,
            "Direct Latency (s)": round(r.direct_latency, 2),
            "Meta Latency (s)": round(r.meta_latency, 2),
            "Additional Latency (s)": round(
                r.meta_latency - r.direct_latency, 2
            ),
        }
        for r in results
    ]
)

summary


,Task,Direct Latency (s),Meta Latency (s),Additional Latency (s)
0,My API is slow. Can you help me fix it?,27.56,36.07,8.51
1,Design a scalable notification system.,14.19,49.09,34.89
2,Make my Lambda function faster.,22.65,42.47,19.81
3,Design a CRM integration.,27.47,41.26,13.78
4,Analyze our customer data.,26.13,30.35,4.22
5,Our database is slow. What should we do?,28.73,37.64,8.91



## 8. Inspect Individual Results

Use this section to compare the actual responses.

When reviewing the results, look for:

### Requirements

- Did the response identify important missing information?
- Did it ask useful clarification questions?
- Did it make reasonable assumptions?

### Technical quality

- Is the proposed solution technically sound?
- Does it address the actual problem?
- Does it consider relevant edge cases?

### Actionability

- Could an engineer act on the response?
- Does it provide concrete next steps?
- Does it avoid unnecessary complexity?

### Prompt quality

For the Meta-Prompting path, inspect whether the generated prompt actually improves the task definition rather than simply making it longer.


In [13]:

def inspect_result(results: list[ExperimentResult], index: int = 0):
    r = results[index]

    print("=" * 80)
    print(f"TASK #{index + 1}")
    print("=" * 80)
    print(r.task)

    print("\n" + "=" * 80)
    print("DIRECT RESULT")
    print("=" * 80)
    print(r.direct_result)

    print("\n" + "=" * 80)
    print("OPTIMIZED META-PROMPT")
    print("=" * 80)
    print(r.optimized_prompt)

    print("\n" + "=" * 80)
    print("META-PROMPTING RESULT")
    print("=" * 80)
    print(r.meta_result)


# Change the index from 0 to inspect another test case.
inspect_result(results, index=0)


TASK #1
My API is slow. Can you help me fix it?

DIRECT RESULT


OPTIMIZED META-PROMPT


META-PROMPTING RESULT




## 9. Manual Evaluation

For the first version of the experiment, manually score each response from 1–5.

Suggested criteria:

| Criterion | Question |
|---|---|
| Goal understanding | Did it understand the user's actual goal? |
| Ambiguity handling | Did it identify important missing information? |
| Assumptions | Were assumptions reasonable and explicit? |
| Technical quality | Is the solution technically sound? |
| Actionability | Can the user act on the answer? |

You can extend the `evaluate_results` function below with your scores.

> These scores are subjective evaluation data. They should be recorded separately from objective metrics such as latency.


In [14]:

# Example structure for manual evaluation.
# Replace the None values with scores from 1-5.

evaluation = []

for i, r in enumerate(results):
    evaluation.append(
        {
            "Task": r.task,
            "Direct Goal Understanding": None,
            "Meta Goal Understanding": None,
            "Direct Ambiguity Handling": None,
            "Meta Ambiguity Handling": None,
            "Direct Technical Quality": None,
            "Meta Technical Quality": None,
            "Direct Actionability": None,
            "Meta Actionability": None,
        }
    )

evaluation_df = pd.DataFrame(evaluation)
evaluation_df


,Task,Direct Goal Understanding,Meta Goal Understanding,Direct Ambiguity Handling,Meta Ambiguity Handling,Direct Technical Quality,Meta Technical Quality,Direct Actionability,Meta Actionability
0,My API is slow. Can you help me fix it?,None,None,None,None,None,None,None,None
1,Design a scalable notification system.,None,None,None,None,None,None,None,None
2,Make my Lambda function faster.,None,None,None,None,None,None,None,None
3,Design a CRM integration.,None,None,None,None,None,None,None,None
4,Analyze our customer data.,None,None,None,None,None,None,None,None
5,Our database is slow. What should we do?,None,None,None,None,None,None,None,None



## 10. Conclusions to Investigate

Do not assume that Meta-Prompting is better simply because it uses an additional LLM call.

The experiment should answer:

1. **Quality:** Does prompt refinement improve the final response?
2. **Ambiguity:** Does the Meta-Refiner identify missing requirements more consistently?
3. **Assumptions:** Does it make better assumptions when clarification is unnecessary?
4. **Cost:** Is the quality improvement worth the additional LLM call?
5. **Latency:** How much additional latency does the Meta LLM introduce?
6. **Use cases:** For which types of underspecified queries does Meta-Prompting appear most useful?

### Expected trade-off

```text
Direct Prompting
    + Lower latency
    + Lower cost
    - Less explicit requirement analysis

Meta-Prompting
    + Better requirement decomposition
    + Potentially better task prompts
    + Better handling of ambiguity
    - Additional LLM call
    - Higher latency
    - Additional token/API cost
```

The goal of this experiment is to determine **when the additional reasoning step is worth the cost**, rather than to assume that Meta-Prompting always improves results.



## 11.  Structured Meta-Refiner

Make the Meta LLM return structured data such as:

```json
{
  "goal": "...",
  "missing_information": ["..."],
  "clarification_needed": false,
  "assumptions": ["..."],
  "optimized_prompt": "..."
}
```

This makes the Meta-Refiner easier to evaluate, debug, and integrate into a production-style application.

You can then experiment with:

```text
User Query
    ↓
Ambiguity Detection
    ↓
Requirement Extraction
    ↓
Prompt Generation
    ↓
Task LLM
    ↓
Critic / Evaluator
    ↓
Final Answer
```


## 12. LLM-as-a-Judge Evaluation

Latency alone does not tell us whether the additional Meta LLM call is worthwhile. This experiment adds an independent evaluator LLM to compare Direct and Meta-Prompting responses on **Response Quality**, **Ambiguity Handling**, and **Task Completeness**, each scored from 1–5.

```text
                  ┌── Direct Answer ──┐
User Task ────────┤                   ├──> Evaluator LLM
                  └── Meta Answer ────┘
                                           ↓
                              Quality / Ambiguity /
                                 Completeness
```

In [24]:
import json
import re
from dataclasses import dataclass

@dataclass
class EvaluationResult:
    task: str
    direct_score: float
    meta_score: float
    direct_ambiguity: float
    meta_ambiguity: float
    direct_completeness: float
    meta_completeness: float
    reasoning: str

EVALUATOR_INSTRUCTIONS = """
You are an impartial evaluator comparing two answers to the same underspecified user request.
Do NOT assume that the longer answer is better.
Evaluate both answers independently using a 1-5 scale:
1 = Very poor, 2 = Poor, 3 = Acceptable, 4 = Good, 5 = Excellent.
Evaluate: Response Quality (technical soundness, usefulness, actionability); Ambiguity Handling (recognizes missing information, asks useful clarification when necessary, makes reasonable assumptions); Task Completeness (covers requirements and important considerations).
Return ONLY valid JSON:
{
  "answer_a": {"quality": 1-5, "ambiguity_handling": 1-5, "completeness": 1-5},
  "answer_b": {"quality": 1-5, "ambiguity_handling": 1-5, "completeness": 1-5},
  "reasoning": "Brief explanation of the evaluation."
}
"""

def evaluate_pair(task: str, answer_a: str, answer_b: str):
    evaluation_prompt = f"""
Evaluate the two answers below.

USER TASK:
{task}

ANSWER A:
{answer_a}

ANSWER B:
{answer_b}

Return ONLY this JSON structure:

{{
  "answer_a": {{
    "quality": 1,
    "ambiguity_handling": 1,
    "completeness": 1
  }},
  "answer_b": {{
    "quality": 1,
    "ambiguity_handling": 1,
    "completeness": 1
  }},
  "reasoning": "Brief explanation, maximum 2 sentences."
}}

Scores must be integers from 1 to 5.
Do not use Markdown.
Do not include ```json.
"""

    response = client.responses.create(
        model=MODEL,
        instructions=EVALUATOR_INSTRUCTIONS,
        input=evaluation_prompt,
        max_output_tokens=1200,
    )

    print("Status:", response.status)
    print("Output:", repr(response.output_text))

    if not response.output_text:
        raise ValueError(
            f"Evaluator returned no text for task: {task}"
        )

    return json.loads(response.output_text)


### Run the Evaluation

The evaluator receives Answer A and Answer B without being told which one came from Direct Prompting or Meta-Prompting. For a more rigorous experiment, randomize answer order and run multiple evaluator trials.

In [25]:
evaluations = []

for i, r in enumerate(results):
    print(f"Evaluating {i + 1}/{len(results)}...")

    try:
        evaluation = evaluate_pair(
            r.task,
            r.direct_result,
            r.meta_result
        )

        evaluations.append(evaluation)

    except Exception as e:
        print(f"⚠️ Evaluation failed: {r.task}")
        print(f"Error: {e}")

        evaluations.append(None)

Evaluating 1/6...
Status: completed
Output: '{\n  "answer_a": {\n    "quality": 1,\n    "ambiguity_handling": 1,\n    "completeness": 1\n  },\n  "answer_b": {\n    "quality": 1,\n    "ambiguity_handling": 1,\n    "completeness": 1\n  },\n  "reasoning": "Both answers are empty and provide no guidance; they fail to address the problem or request clarifications."\n}'
Evaluating 2/6...
Status: completed
Output: '{\n  "answer_a": {\n    "quality": 1,\n    "ambiguity_handling": 1,\n    "completeness": 1\n  },\n  "answer_b": {\n    "quality": 1,\n    "ambiguity_handling": 1,\n    "completeness": 1\n  },\n  "reasoning": "Both answers are empty and provide no design, clarifications, or actionable details."\n}'
Evaluating 3/6...
Status: completed
Output: '{\n  "answer_a": {\n    "quality": 4,\n    "ambiguity_handling": 3,\n    "completeness": 2\n  },\n  "answer_b": {\n    "quality": 1,\n    "ambiguity_handling": 1,\n    "completeness": 1\n  },\n  "reasoning": "Answer A offers accurate, actionabl

In [26]:
successful_evaluations = [
    e for e in evaluations
    if e is not None
]

print(
    f"Successfully evaluated "
    f"{len(successful_evaluations)}/{len(evaluations)} cases."
)

Successfully evaluated 5/6 cases.


In [29]:
# Inspect the evaluation for the first task.
successful_evaluations[0]

{'answer_a': {'quality': 1, 'ambiguity_handling': 1, 'completeness': 1},
 'answer_b': {'quality': 1, 'ambiguity_handling': 1, 'completeness': 1},
 'reasoning': 'Both answers are empty and provide no guidance; they fail to address the problem or request clarifications.'}

## 13. Quality vs. Latency Comparison

Combine objective latency measurements with evaluator scores. The key question is: **Does the quality improvement from Meta-Prompting justify the additional latency and API cost?**

In [30]:
comparison = []

for r, e in zip(results, successful_evaluations):
    direct, meta = e["answer_a"], e["answer_b"]
    direct_avg = (direct["quality"] + direct["ambiguity_handling"] + direct["completeness"]) / 3
    meta_avg = (meta["quality"] + meta["ambiguity_handling"] + meta["completeness"]) / 3
    comparison.append({
        "Task": r.task,
        "Direct Quality": direct["quality"], "Meta Quality": meta["quality"],
        "Direct Ambiguity": direct["ambiguity_handling"], "Meta Ambiguity": meta["ambiguity_handling"],
        "Direct Completeness": direct["completeness"], "Meta Completeness": meta["completeness"],
        "Direct Avg": round(direct_avg, 2), "Meta Avg": round(meta_avg, 2),
        "Direct Latency (s)": round(r.direct_latency, 2), "Meta Latency (s)": round(r.meta_latency, 2),
        "Additional Latency (s)": round(r.meta_latency - r.direct_latency, 2),
    })

comparison_df = pd.DataFrame(comparison)
comparison_df

,Task,Direct Quality,Meta Quality,Direct Ambiguity,Meta Ambiguity,Direct Completeness,Meta Completeness,Direct Avg,Meta Avg,Direct Latency (s),Meta Latency (s),Additional Latency (s)
0,My API is slow. Can you help me fix it?,1,1,1,1,1,1,1.00,1.00,27.56,36.07,8.51
1,Design a scalable notification system.,1,1,1,1,1,1,1.00,1.00,14.19,49.09,34.89
2,Make my Lambda function faster.,4,1,3,1,2,1,3.00,1.00,22.65,42.47,19.81
3,Design a CRM integration.,4,3,5,4,4,3,4.33,3.33,27.47,41.26,13.78
4,Analyze our customer data.,4,2,5,4,4,1,4.33,2.33,26.13,30.35,4.22


In [31]:
metrics = {
    "Direct Quality": comparison_df["Direct Quality"].mean(), "Meta Quality": comparison_df["Meta Quality"].mean(),
    "Direct Ambiguity": comparison_df["Direct Ambiguity"].mean(), "Meta Ambiguity": comparison_df["Meta Ambiguity"].mean(),
    "Direct Completeness": comparison_df["Direct Completeness"].mean(), "Meta Completeness": comparison_df["Meta Completeness"].mean(),
    "Direct Average": comparison_df["Direct Avg"].mean(), "Meta Average": comparison_df["Meta Avg"].mean(),
    "Direct Latency": comparison_df["Direct Latency (s)"].mean(), "Meta Latency": comparison_df["Meta Latency (s)"].mean(),
}
for name, value in metrics.items(): print(f"{name}: {value:.2f}")
quality_improvement = metrics["Meta Average"] - metrics["Direct Average"]
latency_increase = metrics["Meta Latency"] - metrics["Direct Latency"]
print(f"\nQuality improvement: {quality_improvement:+.2f} points")
print(f"Additional latency: {latency_increase:+.2f} seconds")

Direct Quality: 2.80
Meta Quality: 1.60
Direct Ambiguity: 3.00
Meta Ambiguity: 2.20
Direct Completeness: 2.40
Meta Completeness: 1.40
Direct Average: 2.73
Meta Average: 1.73
Direct Latency: 23.60
Meta Latency: 39.85

Quality improvement: -1.00 points
Additional latency: +16.25 seconds


## 14. Interpretation

Do not conclude that Meta-Prompting is better simply because it receives higher scores, or worse simply because it is slower. Use the results to investigate the trade-off between response quality, ambiguity handling, task completeness, latency, and API cost.

A useful follow-up is to group test cases by ambiguity level and determine whether Meta-Prompting provides more value for highly underspecified tasks. The evaluator scores are model-generated judgments; validate a subset with human evaluation and/or multiple evaluator runs for a more rigorous study.